# Full-data MLD hyperparameter optimization

This notebook optimizes the **exact production MLD decoder** independently for the X and Z bases on a deterministic subset of the labelled training data. It does not use cross-validation folds. Each basis has its own tiered MLD parameters, and the objective is the number of incorrect logical bits for that basis, with runtime as a tie-breaker.

The decoder path is identical to `submission_template/mld_ensemble/decoder.py`: forward/reverse MLD tables, `mld_logw`, uncertainty-based refinement, reversed detector columns for the reverse sweep, per-class `np.maximum` log-weight merging, `marginals_logw`, and a 0.5 logical-bit threshold.


In [1]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path("..").resolve()
REPO_ROOT = REPO_ROOT.resolve()
requirements_path = REPO_ROOT / "submission_template" / "requirements.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements_path)])
print("repository:", REPO_ROOT)
print("python:", sys.executable)


repository: /Users/suneg/Dev/QIS-hackathon/qis-my-error
python: /Users/suneg/Dev/QIS-hackathon/qis-my-error/.venv/bin/python



[notice] A new release of pip available: 22.3.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import importlib
import time
from dataclasses import dataclass
from itertools import product

import numpy as np
import stim

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "submission_template"))

from hackathon_2026 import load_kaggle_csv, logical_error_rate, problem_info

# Reload the repository MLD module so a stale notebook import cannot change the API.
sys.modules.pop("mld", None)
mld = importlib.import_module("mld")
for required in ("build_mld", "marginals_logw", "mld_logw"):
    if not hasattr(mld, required):
        raise ImportError(f"loaded unexpected MLD module {getattr(mld, '__file__', None)}")
build_mld = mld.build_mld
marginals_logw = mld.marginals_logw
mld_logw = mld.mld_logw

DATA_PATH = REPO_ROOT / "data" / "train" / "train.csv"
SEED = 42


## Load a deterministic labelled subset

To keep repeated MLD evaluations practical, this loads a fixed, reproducible subset from each basis. Increase `SUBSET_PER_BASIS` for a more precise final comparison after the search has narrowed the candidates.


In [3]:
SUBSET_PER_BASIS = 20000

def load_full_data(path: Path, subset_per_basis: int | None, seed: int):
    data = load_kaggle_csv(str(path))
    if "observable_flips" not in data:
        raise ValueError("full-data optimization requires observable_flips labels")
    result = {}
    for basis in ("X", "Z"):
        rows = np.flatnonzero(data["basis"] == basis)
        if len(rows) == 0:
            raise ValueError(f"no labelled {basis} rows found")
        rng = np.random.default_rng(seed + (0 if basis == "X" else 1))
        rng.shuffle(rows)
        if subset_per_basis is not None:
            rows = rows[:min(subset_per_basis, len(rows))]
        result[basis] = {
            "events": np.ascontiguousarray(data["detector_events"][rows], dtype=np.uint8),
            "labels": np.ascontiguousarray(data["observable_flips"][rows], dtype=bool),
        }
    return result


def load_problem_data(folder: Path):
    result = {}
    for basis in ("X", "Z"):
        circuit = stim.Circuit.from_file(str(folder / f"{basis}_circuit.stim"))
        result[basis] = problem_info(circuit)
    return result

basis_data = load_full_data(DATA_PATH, SUBSET_PER_BASIS, SEED)
problems = load_problem_data(DATA_PATH.parent)
for basis in ("X", "Z"):
    print(f"{basis}: shots={len(basis_data[basis]['events'])}, events={basis_data[basis]['events'].shape}, labels={basis_data[basis]['labels'].shape}")


X: shots=20000, events=(20000, 70), labels=(20000, 4)
Z: shots=20000, events=(20000, 70), labels=(20000, 4)


## Exact production-equivalent decoder and objective

`decode_candidate` is deliberately written to match `decoder.decode_marginals`. A candidate tier is `(direction, eps, max_states, threshold)`, where direction 0 uses normal detector order and direction 1 uses reversed detector order.


In [4]:
TABLE_CACHE = {}
EVALUATION_CACHE = {}


def get_tables(basis: str, hexp: float):
    key = (basis, float(hexp))
    if key not in TABLE_CACHE:
        info = problems[basis]
        TABLE_CACHE[key] = (
            build_mld(info["H"], info["L"], info["p"], hexp=hexp),
            build_mld(info["H"][::-1].copy(), info["L"], info["p"], hexp=hexp),
        )
    return TABLE_CACHE[key]


def decode_candidate(tables, detector_events, tiers, compact_every):
    D = np.ascontiguousarray(np.asarray(detector_events, dtype=bool))
    events = (D, np.ascontiguousarray(D[:, ::-1]))
    logw = None
    for direction, eps, max_states, threshold in tiers:
        if logw is None:
            logw = mld_logw(
                tables[direction], events[direction], eps=eps,
                max_states=int(max_states), compact_every=int(compact_every),
            )
            continue
        selected = np.flatnonzero(
            (np.abs(marginals_logw(logw) - 0.5) < 0.5 - threshold).any(axis=1)
        )
        if len(selected):
            refined = mld_logw(
                tables[direction], events[direction][selected], eps=eps,
                max_states=int(max_states), compact_every=int(compact_every),
            )
            logw[selected] = np.maximum(logw[selected], refined)
    return (marginals_logw(logw) > 0.5).astype(np.uint8)


def candidate_key(candidate):
    return (candidate["tiers"], float(candidate["hexp"]), int(candidate["compact_every"]))


def evaluate_candidate(basis: str, candidate, verbose=True):
    key = (basis, candidate_key(candidate))
    if key in EVALUATION_CACHE:
        return EVALUATION_CACHE[key]
    started = time.perf_counter()
    tables = get_tables(basis, candidate["hexp"])
    predictions = decode_candidate(
        tables,
        basis_data[basis]["events"],
        candidate["tiers"],
        candidate["compact_every"],
    )
    labels = basis_data[basis]["labels"]
    wrong = int(np.count_nonzero(predictions.astype(bool) != labels))
    result = {
        "basis": basis,
        "candidate": candidate,
        "wrong": wrong,
        "bits": int(predictions.size),
        "ler": logical_error_rate(predictions, labels),
        "seconds": time.perf_counter() - started,
    }
    EVALUATION_CACHE[key] = result
    if verbose:
        print(f"{basis}: wrong={wrong} LER={result['ler']:.8f} time={result['seconds']:.2f}s candidate={candidate}")
    return result


def objective_key(result):
    return (result["wrong"], result["seconds"])


## Baseline and bounded coordinate search

The baseline matches the current production configuration. Coordinate search evaluates both neighboring values for each numeric parameter, moves to an improving neighbor, and repeats until a full round makes no move or the safety limits are reached. `hexp` remains fixed at the baseline value; X additionally searches `compact_every` independently, while Z searches its tier parameters.


In [5]:
BASELINE = {
    "X": {
        "tiers": (
            (0, 6.666666666666667e-5, 3072, 0.0),
            (1, 4.5e-5, 3072, 0.125),
            (0, 4.5e-5, 3072, 0.25),
        ),
        "hexp": 0.6,
        "compact_every": 32,
    },
    "Z": {
        "tiers": (
            (0, 6.666666666666667e-5, 3072, 0.0),
            (1, 4.5e-5, 3072, 0.125),
            (0, 4.5e-5, 3072, 0.25),
        ),
        "hexp": 0.6,
        "compact_every": 32,
    },
}
COORDINATE_MAX_ROUNDS = 30
COORDINATE_MAX_STEPS_PER_PARAMETER = 8


def point_from_candidate(candidate):
    first, second, third = candidate["tiers"]
    return {
        "first_eps": first[1], "first_states": first[2],
        "second_eps": second[1], "second_states": second[2],
        "second_threshold": second[3],
        "third_eps": third[1], "third_states": third[2],
        "third_threshold": third[3], "hexp": candidate["hexp"],
        "compact_every": candidate["compact_every"],
    }


def candidate_from_point(point, directions=(0, 1, 0)):
    return {
        "tiers": (
            (directions[0], point["first_eps"], int(point["first_states"]), 0.0),
            (directions[1], point["second_eps"], int(point["second_states"]), point["second_threshold"]),
            (directions[2], point["third_eps"], int(point["third_states"]), point["third_threshold"]),
        ),
        "hexp": float(point["hexp"]),
        "compact_every": int(point["compact_every"]),
    }


def neighbors(point, name):
    lower, upper = point.copy(), point.copy()
    value = point[name]
    if name in ("first_eps", "second_eps", "third_eps"):
        lower[name] = max(1e-7, value / 1.5)
        upper[name] = min(1e-2, value * 1.5)
    elif name in ("first_states", "second_states", "third_states"):
        lower[name] = max(128, value - 256)
        upper[name] = min(32768, value + 256)
    elif name in ("second_threshold", "third_threshold"):
        lower[name] = max(0.0, value - 0.025)
        upper[name] = min(0.95, value + 0.025)
    elif name == "compact_every":
        lower[name] = max(4, value - 8)
        upper[name] = min(128, value + 8)
    return lower, upper


def coordinate_search(basis, start=None):
    point = point_from_candidate(start or BASELINE[basis])
    current = evaluate_candidate(basis, candidate_from_point(point))
    names = (
        "first_eps", "first_states", "second_eps", "second_states", "second_threshold",
        "third_eps", "third_states", "third_threshold",
    )
    if basis == "X":
        names += ("compact_every",)
    history = []
    for round_index in range(1, COORDINATE_MAX_ROUNDS + 1):
        moved = False
        for name in names:
            for _ in range(COORDINATE_MAX_STEPS_PER_PARAMETER):
                lower, upper = neighbors(point, name)
                candidates = [(point, current)]
                if lower[name] != point[name]:
                    candidates.append((lower, evaluate_candidate(basis, candidate_from_point(lower))))
                if upper[name] != point[name]:
                    candidates.append((upper, evaluate_candidate(basis, candidate_from_point(upper))))
                best_point, best_result = min(candidates, key=lambda item: objective_key(item[1]))
                if objective_key(best_result) >= objective_key(current):
                    break
                old = point[name]
                point, current = best_point, best_result
                moved = True
                history.append((round_index, name, old, point[name], current["wrong"], current["ler"]))
                print(f"{basis}: move {name}: {old} -> {point[name]}, wrong={current['wrong']}")
        if not moved:
            print(f"{basis}: coordinate search converged")
            break
    return candidate_from_point(point), current, history

coordinate_results = {}
for basis in ("X", "Z"):
    candidate, result, history = coordinate_search(basis)
    coordinate_results[basis] = (candidate, result, history)
    print(f"{basis} coordinate candidate:", candidate)
    print(f"{basis} coordinate result:", result)


X: wrong=2726 LER=0.03407500 time=12.08s candidate={'tiers': ((0, 6.666666666666667e-05, 3072, 0.0), (1, 4.5e-05, 3072, 0.125), (0, 4.5e-05, 3072, 0.25)), 'hexp': 0.6, 'compact_every': 32}
X: wrong=2722 LER=0.03402500 time=5.39s candidate={'tiers': ((0, 4.4444444444444447e-05, 3072, 0.0), (1, 4.5e-05, 3072, 0.125), (0, 4.5e-05, 3072, 0.25)), 'hexp': 0.6, 'compact_every': 32}
X: wrong=2731 LER=0.03413750 time=4.32s candidate={'tiers': ((0, 0.0001, 3072, 0.0), (1, 4.5e-05, 3072, 0.125), (0, 4.5e-05, 3072, 0.25)), 'hexp': 0.6, 'compact_every': 32}
X: move first_eps: 6.666666666666667e-05 -> 4.4444444444444447e-05, wrong=2722
X: wrong=2725 LER=0.03406250 time=6.12s candidate={'tiers': ((0, 2.962962962962963e-05, 3072, 0.0), (1, 4.5e-05, 3072, 0.125), (0, 4.5e-05, 3072, 0.25)), 'hexp': 0.6, 'compact_every': 32}
X: wrong=2722 LER=0.03402500 time=5.04s candidate={'tiers': ((0, 4.4444444444444447e-05, 2816, 0.0), (1, 4.5e-05, 3072, 0.125), (0, 4.5e-05, 3072, 0.25)), 'hexp': 0.6, 'compact_every

## Joint local refinement and decoder-structure search

Coordinate search can miss interactions between parameters. This stage evaluates 3x3 neighborhoods for all three epsilon/state pairs, then tests alternate direction orders and a three-pass structure.


In [ ]:
def three_values(value, name):
    lower, upper = neighbors({name: value}, name)
    return (lower[name], value, upper[name])


def joint_candidates(base):
    point = point_from_candidate(base)
    for values in product(
        three_values(point["first_eps"], "first_eps"),
        three_values(point["first_states"], "first_states"),
        three_values(point["second_eps"], "second_eps"),
        three_values(point["second_states"], "second_states"),
        three_values(point["third_eps"], "third_eps"),
        three_values(point["third_states"], "third_states"),
    ):
        p = point.copy()
        p.update(
            first_eps=values[0], first_states=values[1],
            second_eps=values[2], second_states=values[3],
            third_eps=values[4], third_states=values[5],
        )
        yield candidate_from_point(p)


final_results = {}
for basis in ("X", "Z"):
    coordinate_candidate, coordinate_result, _ = coordinate_results[basis]
    joint_results = [
        evaluate_candidate(basis, candidate)
        for candidate in joint_candidates(coordinate_candidate)
    ]
    joint_best = min(joint_results, key=objective_key)
    tiers = joint_best["candidate"]["tiers"]
    structure_candidates = [
        joint_best["candidate"],
        {**joint_best["candidate"], "tiers": tuple(reversed(tiers))},
        {
            **joint_best["candidate"],
            "tiers": (tiers[1], tiers[2], tiers[0]),
        },
        {
            **joint_best["candidate"],
            "tiers": (tiers[2], tiers[0], tiers[1]),
        },
    ]
    structure_results = [evaluate_candidate(basis, candidate) for candidate in structure_candidates]
    final_results[basis] = min(structure_results, key=objective_key)
    print(f"{basis} joint best:", joint_best)
    print(f"{basis} final best candidate:", final_results[basis]["candidate"])
    print(f"{basis} final best result:", final_results[basis])


X: wrong=2725 LER=0.03406250 time=7.24s candidate={'tiers': ((0, 2.962962962962963e-05, 2560, 0.0), (1, 2e-05, 1792, 0.125), (0, 4.5e-05, 2560, 0.25)), 'hexp': 0.6, 'compact_every': 32}
X: wrong=2725 LER=0.03406250 time=7.74s candidate={'tiers': ((0, 2.962962962962963e-05, 2560, 0.0), (1, 2e-05, 1792, 0.125), (0, 4.5e-05, 2816, 0.25)), 'hexp': 0.6, 'compact_every': 32}
X: wrong=2725 LER=0.03406250 time=7.88s candidate={'tiers': ((0, 2.962962962962963e-05, 2560, 0.0), (1, 2e-05, 1792, 0.125), (0, 4.5e-05, 3072, 0.25)), 'hexp': 0.6, 'compact_every': 32}
X: wrong=2725 LER=0.03406250 time=6.99s candidate={'tiers': ((0, 2.962962962962963e-05, 2560, 0.0), (1, 2e-05, 1792, 0.125), (0, 6.75e-05, 2560, 0.25)), 'hexp': 0.6, 'compact_every': 32}
X: wrong=2725 LER=0.03406250 time=6.69s candidate={'tiers': ((0, 2.962962962962963e-05, 2560, 0.0), (1, 2e-05, 1792, 0.125), (0, 6.75e-05, 2816, 0.25)), 'hexp': 0.6, 'compact_every': 32}
X: wrong=2725 LER=0.03406250 time=6.42s candidate={'tiers': ((0, 2.9

## Final report

The result below is the best full-data empirical configuration found by this notebook. Re-run the final candidate on a fresh holdout before treating a very small improvement as robust.


In [ ]:
print("BEST PRODUCTION PARAMETERS BY BASIS")
print("Tier values and compact_every are searched; hexp remains fixed at the baseline value.")
for basis in ("X", "Z"):
    result = final_results[basis]
    print(f"{basis}: MLD_TIERS =", result["candidate"]["tiers"])
    print(f"{basis}: MLD_COMPACT_EVERY =", result["candidate"]["compact_every"])
    print(f"{basis}: MLD_HEXP =", result["candidate"]["hexp"])
    print(f"{basis}: wrong logical bits =", result["wrong"])
    print(f"{basis}: total logical bits =", result["bits"])
    print(f"{basis}: LER =", result["ler"])
    print(f"{basis}: runtime seconds =", result["seconds"])
print("evaluated unique candidates =", len(EVALUATION_CACHE))
